# 3 · What is a CoefficientFunction?

Almost everything with a value in space is, in NGSolve, a **`CoefficientFunction`**
(CF): 
* a material parameter,
* the right-hand side,
* the computed solution.

The single idea that unlocks all of them:

> **A CF is a function whose argument is a _mapped integration point_.**

You rarely call a CF with a plain $(x,y)$. Instead it is typically evaluated *for* you, at a **mapped integration point** within element loops:

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "anywidget"], check=True)

![Reference element, the element map Φ_T, and the resulting mapped point in a
physical (curved) element that belongs to a region.](data/cf-mapping.png)

* A reference point $\hat x$ on the reference element $\hat T$
* sent by the **element trafo** $\Phi_T$

In [ ]:
from ngsolve.meshes import *
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw
from math import pi
import numpy as np

## 1. The argument of a `CF`: a mapped integration point

A mapped integration point carries everything a `CF` could ask for:

* a **reference coordinate** $\hat x$ on the reference element;
* an **element transformation** $\Phi_T$, and through it
  * the **element kind** — a volume or a surface element,
  * the **region** it belongs to — which material / which boundary,
  * the **transformation data** — the Jacobian $\partial\Phi_T$, and
  * the **world coordinate** $x=\Phi_T(\hat x)$.

Different contexts lean on different aspects of this one unifying interface:

* **world coordinates**,
* **local** point of view (FE stuff; often tied to trafo)
* **region-wise** expressions,
* **combinations**

Let us meet each aspect with a tiny example.

And besides *where* it is evaluated, a `CF` also has a **value type** — its image
is **scalar**, **vector** or **matrix**-valued, readable from `.dim` / `.dims`:

In [ ]:
print("value type:  x →", x.dim, "component (scalar) |",
      "CF((x,y)) →", CF((x, y)).dim, "components (vector) |",
      "JacobianMatrix(2).dims =", tuple(specialcf.JacobianMatrix(2).dims), "(matrix)")

`specialcf`? NGSolve provides some convenient `CF`s 

In [ ]:
print([m for m in dir(specialcf) if not m.startswith('_')])

### 1a · World coordinates

The coordinates `x`, `y`, `z` are CFs: each applies the element map to the
reference point and returns the **world** coordinate. Anything built from them
(like `sin(pi*x)`) is a field you can evaluate anywhere.

In [ ]:
left  = WorkPlane().MoveTo(0, 0).Rectangle(1, 1).Face(); left.name  = "left"
right = WorkPlane().MoveTo(1, 0).Rectangle(1, 1).Face(); right.name = "right"
strip = Glue([left, right])
strip.edges.Min(X).name = "west"; strip.edges.Max(X).name = "east"
mesh2 = Mesh(OCCGeometry(strip, dim=2).GenerateMesh(maxh=0.3))

# Mesh Points:
mp1 = mesh2(0.3, 0.7); mp2 = mesh2(1.6, 0.2)
from ngsolve.fem import *
# Mapped Integration Points:
mip1 = BaseMappedIntegrationPoint(mp1)

print(np.array(mp1.pnt), "->", np.array(mip1.point))

print("x is the world coordinate:  x(0.3, 0.7) =", x(mp1), "   x(1.6, 0.2) =", x(mp2))
Draw(x, mesh2, "x — the world coordinate")

### 1b · Regions: volumes and surfaces, in 2D and 3D

Because the point knows its **element**, it knows the element's **region** and
whether it is a **volume** or a **surface** element. A `MaterialCF` returns a
different value per material; a `BoundaryCF` does the same on the boundary. In 2D:

In [ ]:
matCF = mesh2.MaterialCF({"left": 0.2, "right": 0.4})*x
print("region-wise (2D): ∫ on 'left' =", Integrate(matCF, mesh2, definedon=mesh2.Materials("left")), "∫ on 'right' =", Integrate(matCF, mesh2, definedon=mesh2.Materials("right")))
Draw(matCF, mesh2, "matCF — a different value per material", deformation=True)

We see: you can multiply `CF`s (of different origin) resulting in a new `CF`.
It has an evaluation tree where each leaf is a specific "basic" `CF`:

In [ ]:
print(matCF)

The same region-wise definition works in **3D**, where a CF distinguishes **volumetric** regions and
**surface** regions alike — two boxes glued into a `bottom` and a `top` material,
with named faces `floor` and `lid`:

In [ ]:
bottom = Box(Pnt(0, 0, 0), Pnt(1, 1, .5)); bottom.solids.name = "bottom"
top    = Box(Pnt(0, 0, .5), Pnt(1, 1, 1)); top.solids.name = "top"
cube = Glue([bottom, top])
cube.faces.Min(Z).name, cube.faces.Max(Z).name = "floor", "lid"
mesh3 = Mesh(OCCGeometry(cube).GenerateMesh(maxh=0.4))

kappa = mesh3.MaterialCF({"bottom": 1.0, "top": 5.0})              # a volumetric CF
load  = mesh3.BoundaryCF({"floor": 2.0, "lid": 9.0}, default=0.0)  # a surface CF
print("volumetric CF:  mean κ on 'top'   =",
      Integrate(kappa, mesh3, definedon=mesh3.Materials("top")) / Integrate(CF(1), mesh3, definedon=mesh3.Materials("top")))
print("surface    CF:  ∫ load ds on 'lid' =", Integrate(load, mesh3.Boundaries("lid")))
# Pass the clipping plane as a Draw kwarg (not via settings): this sets the scene's
# `clipping` flag, which the webgui uses to switch the plane ON at load — and
# `function=True` draws the field on the cut, so the cross-section shows κ directly,
# with no user interaction. (settings={"Clipping": …} only pre-fills the GUI control
# and still needs a click.)
clip3d = {"function": True, "x": 0, "y": 1, "z": 0, "dist": 0}
gf_kappa = GridFunction(L2(mesh3, order=0)); gf_kappa.Set(kappa)
view = dict(euler_angles=[-53.411050822757815,9.354250074623367,47.265921621581114])
Draw(gf_kappa, mesh3, "κ", clipping=clip3d, **view)
Draw(load, mesh3, "load", draw_vol=False, draw_surf=True, clipping=clip3d, **view)

:::{dropdown} 🔧 How `MaterialCF` / `BoundaryCF` work under the hood
Every element carries an integer **region index**; `mesh.GetMaterials()` and
`mesh.GetBoundaries()` are simply the **names** in that index order. So a
`MaterialCF({name: value})` is sugar for a **domain-wise list** `CF([...])`,
one entry per region index — and you can build it by hand from `GetMaterials()`.
Boundaries work the same way, indexed by `GetBoundaries()`. Expand the code cell
below to see the masks and rebuild `κ` yourself.
:::

In [ ]:
print("materials by region index :", mesh3.GetMaterials())
print("boundaries by region index:", mesh3.GetBoundaries())

# MaterialCF({...}) == a domain-wise list CF, ordered by the material index:
values = {"bottom": 1.0, "top": 5.0}
kappa_by_hand = CF([values[name] for name in mesh3.GetMaterials()]) # -> CF([1,5])
print("by-hand list CF equals MaterialCF? ",
      Integrate((kappa_by_hand - kappa)**2, mesh3) < 1e-20)

### 1c · Local coordinates

And, most directly, a CF can hand back the **reference coordinate** it was given —
`specialcf.xref`. That is all it is:

In [ ]:
xref = specialcf.xref(2)
print("reference coordinate at world point (1.6, 0.2):",
      tuple(round(v, 3) for v in xref(mesh2(1.6, 0.2))))
Draw(xref[0], mesh2, "ξ = xref[0] — the local coordinate (per element)")

This local view is where finite elements begin: on a triangle the linear (**P1**)
**shape functions** are the **barycentric coordinates**
$$\lambda=(1-\xi-\eta,\ \xi,\ \eta)$$ 
— each $1$ at one corner, $0$ at the others.

## 2. A hat function — a global basis function as a CF

We now want a single CF that is
* $1$ at one chosen mesh vertex $V$ and
* $0$ at every other vertex**
* (continuous in between)

the **hat function** $\varphi_V$. 

On each element it is the barycentric coordinate of $V$ — *if* $V$ is one of that element's corners.

We will want **both** the hat and its gradient, assembled the same way from the
barycentric coordinates $\lambda_i$ and their gradients:

$$ \varphi_V = \sum_{i=0}^{2} \lambda_i\,\texttt{is\_vertex}_i(V),
   \qquad
   \nabla\varphi_V = \sum_{i=0}^{2} \nabla\lambda_i\,\texttt{is\_vertex}_i(V). $$

A CF cannot read vertex numbers, so the indicator $\texttt{is\_vertex}_i(V)$ —
"is corner $i$ of this element the vertex $V$?" — is built once (and folded away):

In [ ]:
# --- HIDDEN: construction of is_vertex(i,V) ---------------------------------
# Store, per element, the global numbers of its three corners IN REFERENCE ORDER
# (so corner[i] is the vertex where the barycentric λ_i equals 1), found by
# mapping the reference vertices through the element transformation and matching
# world coordinates.
mesh = MakeStructured2DMesh(quads=False, nx=4, ny=4)        # one coarse mesh, used throughout
_cells = L2(mesh, order=0)
corner = [GridFunction(_cells) for _ in range(3)]
_refvtx = IntegrationRule([(0, 0), (1, 0), (0, 1)], [1/6, 1/6, 1/6])
for el in mesh.Elements(VOL):
    W = np.array(CF((x, y))(mesh.GetTrafo(el)(_refvtx))).reshape(3, -1)[:, :2]
    pos = {v.nr: np.array(mesh[v].point) for v in el.vertices}
    d = _cells.GetDofNrs(el)[0]
    for i in range(3):
        corner[i].vec[d] = next(nr for nr, c in pos.items() if np.allclose(c, W[i]))

def _abs(e): return IfPos(e, e, -e)
def is_vertex(i, V): return IfPos(0.5 - _abs(corner[i] - V), 1, 0)   # corner i == V ? 1 : 0

The **barycentric coordinates** are the local shape functions, straight from
`xref`. With them the hat is a one-liner — let us draw it (deformation scaled to
`0.25`, mesh edges off, for a clean tent):

In [ ]:
lam = [1 - xref[0] - xref[1], xref[0], xref[1]]       # barycentric = local shape functions
def hat(V): return sum(lam[i] * is_vertex(i, V) for i in range(3)) #selects the barycentric coord to global vertex V (if exists)

Vc = min(mesh.vertices, key=lambda v: sum((c - 0.5)**2 for c in mesh[v].point)).nr # choose vertex closest to (0.5,0.5)
Draw(hat(Vc), mesh, "a hat function φ_V — built as a CoefficientFunction", deformation=True, settings={"deformation": 0.25, "Objects": {"Edges": False}})

### Pitfall — the gradient is *not* `hat.Diff`

NGSolve can form derivatives. But be careful what to expect!

We still need $\nabla\varphi_V$. The naive reflex, `hat(Vc).Diff(x)`, returns
**zero**:

In [ ]:
print("hat(Vc).Diff(x):   ∫ |·|² =", round(Integrate(hat(Vc).Diff(x)**2, mesh), 8))

Why? Because **a CF is an evaluation tree**, and `.Diff(x)` differentiates *that
tree* — and only that tree. Take a familiar field, $u_{ex}=\sin(\pi x)\sin(\pi y)$,
and print its tree:

In [ ]:
uex = sin(pi*x) * sin(pi*y)
print(uex)                           # the raw evaluation tree (verbose, but a tree)

Schematically that is a small tree, and `.Diff(x)` walks it (product + chain
rule), differentiating every `x` it meets:
```text
         ·                       .Diff(x)           ·
        / \          ────────────────────▶        /   \
     sin   sin                            π·cos(πx)   sin(πy)
      |     |
     π·x   π·y
```
so `uex.Diff(x)` $= \pi\cos(\pi x)\sin(\pi y)$. 

The **hat's** tree, by contrast,
is built from `xref` and per-element **constants** (the `corner` grid functions
inside `is_vertex`); its depth even grows with the mesh, so here just a sketch:
```text
        Σ over the 3 corners
             |      
             ·
          /     \
      λ_i         is_vertex_i(V)
       |               |
   xref (ξ,η)    per-element constants      ← no symbolic x or y anywhere
```
There is **no `x` in this tree** → `.Diff(x)` is identically `0`. The hat *does*
vary in space, but only through the **element map**, which the tree never sees.
The true gradient must therefore come from the **mapping** itself.

### The gradient, from the Jacobian

The world gradient of a barycentric coordinate is $\nabla\lambda_i =
J^{-\top}\,\hat\nabla\lambda_i$, where the **reference** gradients
$\hat\nabla\lambda_i$ are *constants* and $J=\partial\Phi_T$ is the element
Jacobian — a CF, `specialcf.JacobianMatrix`. No `.Diff` involved:

In [ ]:
_gref = [CF((-1, -1)), CF((1, 0)), CF((0, 1))]            # ∇_ref λ_i  (constant)
grad_lam = [Inv(specialcf.JacobianMatrix(2)).trans * g for g in _gref]   # world gradients
def grad_hat(V): return sum((grad_lam[i] * is_vertex(i, V) for i in range(3)), CF((0, 0)))
Draw(grad_hat(Vc), mesh, "∇φ_V", vectors={"grid_size": 30})

## 3. Solving a PDE on a triangular mesh — by hand

A finite element solution is a **weighted sum of hats**, $u_h=\sum_i c_i\varphi_i$.
The (Galerkin) recipe for $-\Delta u = f$ with $u=0$ on the boundary: find the
weights so that, *tested against every interior hat $\varphi_j$*,
$$ \int_\Omega \nabla u_h\cdot\nabla\varphi_j \,dx = \int_\Omega f\,\varphi_j\,dx. $$
Both sides are **integrals of coefficient functions** — exactly what `Integrate`
evaluates. With homogeneous Dirichlet data the unknowns are the **interior**
vertices. We assemble the (small, dense) system naively — *every* interior hat
against *every* other — and solve it with numpy.

In [ ]:
interior = [v.nr for v in mesh.vertices if 1e-9 < mesh[v].point[0] < 1 - 1e-9 and 1e-9 < mesh[v].point[1] < 1 - 1e-9]
print(f"{mesh.nv} vertices,  {len(interior)} interior unknowns")

def solve(f):
    n = len(interior)
    gh = {V: grad_hat(V) for V in interior}
    K = np.array([[Integrate(gh[Vi]*gh[Vj], mesh, order=2) for Vj in interior] for Vi in interior])
    b = np.array([Integrate(f*hat(Vi), mesh, order=5) for Vi in interior])
    c = np.linalg.solve(K, b)
    return sum(float(c[a])*hat(Vi) for a, Vi in enumerate(interior))   # u_h, as a CF

u_h = solve(CF(1.0)) 
Draw(u_h, mesh, "u_h", deformation=True)

## 4. Measuring accuracy — `Integrate` and a constructive `Diff`

Is that solution any good? 

The **method of manufactured solutions**: 
*pick* the answer $u_{ex}$ and let `Diff` hand us the load $f=-\Delta u_{ex}$ that produces it. 

In [ ]:
f = -(uex.Diff(x).Diff(x) + uex.Diff(y).Diff(y))          # = -Δuex, by symbolic Diff
u_h = solve(f)
print("L2 error ‖u_h − u_ex‖ =", round(sqrt(Integrate((u_h - uex)**2, mesh, order=6)), 4))

## 5. Why hand it to NGSolve

Our by-hand machine works, but it is naive in ways that matter:

* it **ignores locality** — hats overlap only with their neighbours, yet we
  integrated *every* dof against *every* other, an $\mathcal{O}(N^2)$ dense
  assembly where the real matrix is **sparse**;
* the Dirichlet bookkeeping, the reference-order vertex matching, the quadrature
  orders — all hand-rolled and easy to get subtly wrong;
* and it is fixed to P1 on triangles.

An NGSolve **`FESpace`** does all of this maturely and fast: the hat functions,
their gradients, locality and sparsity, boundary dofs, arbitrary order and
element type. As a farewell to hand-made basis functions, here is a single `H1`,
order-1 basis function — the very hat we built, now handed to us by the space
(activate one degree of freedom and draw):

In [ ]:
fes = H1(mesh, order=1)
gfb = GridFunction(fes)
gfb.vec[:] = 0
gfb.vec[fes.GetDofNrs(NodeId(VERTEX, Vc))[0]] = 1     # one basis function "switched on"
Draw(gfb, mesh, "H1hat", deformation=True)

Next: **finite element spaces** in their own right — the families of shape
functions NGSolve provides, and what each one keeps continuous.

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("02-geometry", "2 · Creating Geometry and Meshes")
    _next = ("04-fespaces", "4 · A zoo of finite element spaces")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))